# 第9回　仮説検定(2)：分散分析（ANOVA）
## ―― 3つ以上を比べるとき、t検定を繰り返してはいけない

統計学Ⅰ（B）　／　北星学園大学

注目は ――

> 検定を**繰り返す**と、偽陽性（まぐれの「有意」）が**勝手に増えていく。**

### フック

> 3つの学部（経済・文・社福）で、テスト点に差があるか調べたい。
>
> 「じゃあ、**経済vs文・文vs社福・経済vs社福** と、全ペアで t検定すればいいよね？」

――この『全ペアでt検定』が、実は危ない。なぜかを確かめる。

In [ ]:
!pip install -q japanize-matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib  # noqa: F401
from scipy import stats
from itertools import combinations

URL = "https://aonoa68.github.io/toukei-1/data/hokushin_students.csv"
try:
    df = pd.read_csv(URL)
except Exception:
    R = np.random.default_rng(2026); N = 400
    disc = R.normal(0,1,N); apt = R.normal(0,1,N)
    gk = R.choice(["経済学部","文学部","社会福祉学部"], N, p=[.40,.35,.25])
    gke = np.select([gk=="経済学部",gk=="文学部",gk=="社会福祉学部"],[2.,-1.,-1.])
    gen = R.choice(["女","男","回答しない"], N, p=[.55,.43,.02])
    bh = np.where(gen=="男",171.,158.); bh=np.where(gen=="回答しない",165.,bh)
    height = bh + R.normal(0,6,N)
    alone = R.random(N)<.35
    com = np.clip(np.where(alone,R.normal(20,8,N),R.normal(55,25,N)),5,None)
    slp = 7+.5*disc-.005*(com-30)+R.normal(0,.8,N)
    sns = np.clip(3-.8*disc+R.normal(0,1.,N),0,None)
    std = np.clip(1.5+.7*disc+.2*apt+R.normal(0,.6,N),0,None)
    pt  = np.clip(np.where(alone,R.normal(18,6,N),R.normal(10,6,N)),0,None)
    att = np.clip(82+7*disc+R.normal(0,5,N),0,100)
    bf  = np.clip(np.round(4+1.6*disc+R.normal(0,1.,N)),0,7)
    test= np.clip(55+7*apt+4*std+1.2*(slp-7)-1.5*sns+gke+R.normal(0,6,N),0,100)
    inc = R.lognormal(np.log(550),.45,N)
    df = pd.DataFrame({"学生ID":[f"26B{i+1:04d}" for i in range(N)],"学部":gk,"性別":gen,
        "身長cm":np.round(height,1),"一人暮らし":np.where(alone,"はい","いいえ"),
        "通学時間min":np.round(com).astype(int),"睡眠時間h":np.round(slp,1),"SNS時間h":np.round(sns,1),
        "勉強時間h":np.round(std,1),"アルバイト時間week":np.round(pt).astype(int),"出席率":np.round(att).astype(int),
        "朝食日数week":bf.astype(int),"テスト点":np.round(test).astype(int),"世帯年収万円":np.round(inc).astype(int)})
    df.loc[3,"世帯年収万円"]=12000; df.loc[88,"世帯年収万円"]=9500; df.loc[7,"身長cm"]=1710.0
    df.loc[15,"睡眠時間h"]=np.nan; df.loc[42,"睡眠時間h"]=np.nan; df.loc[101,"通学時間min"]=np.nan
print("準備OK")

---
## 1. なぜ「繰り返し」がダメなのか

1回のt検定は「本当は差がないのに、まぐれで有意」と誤る確率を **5%（α）** 許している。

では検定を**何回も**やったら？　毎回5%の落とし穴があるのだから、回数が増えるほど「どれか1つはまぐれで有意」になりやすい。これを確かめる。

---
## 2. 多重比較の罠（シミュレーション）

**本当はすべて同じ母集団（＝差がまったくない）** の k 個の群を作り、全ペアで t検定する。「最低1つでも有意（p<0.05）」になってしまう割合を数える。差はないのだから、本来5%であってほしい。

In [ ]:
rng = np.random.default_rng(2026)
ks = [2, 3, 5, 10]
偽陽性率 = []
for k in ks:
    fp = 0
    for _ in range(3000):
        群 = [rng.normal(50, 10, 30) for _ in range(k)]   # 全部おなじ＝差なし
        もし1つでも有意 = any(stats.ttest_ind(群[i], 群[j])[1] < 0.05
                          for i, j in combinations(range(k), 2))
        fp += もし1つでも有意
    率 = fp/3000*100
    偽陽性率.append(率)
    print(f"群数 k={k:2d}（{k*(k-1)//2:2d}ペア）: 最低1つ有意になった割合 = {率:3.0f}%（本来は5%のはず）")

In [ ]:
plt.figure(figsize=(6.5,3.6))
bars = plt.bar([str(k) for k in ks], 偽陽性率, color="#e8503a")
plt.axhline(5, color="gray", ls="--", label="本来あるべき 5%")
for b,r in zip(bars,偽陽性率): plt.text(b.get_x()+b.get_width()/2, r+1.5, f"{r:.0f}%", ha="center")
plt.xlabel("群の数 k"); plt.ylabel("まぐれで有意になった割合")
plt.title("差がないのに、群を増やすほど偽陽性が増える"); plt.legend(); plt.show()

**差はまったくないのに**、群が増える（＝ペア＝検定回数が増える）ほど「最低1つ有意」が増える。10群なら6割以上がまぐれの「発見」だ。

だから3群以上を「全ペアt検定」してはいけない。**検定の回数を増やすほど、偽の発見が紛れ込む。**

---
## 3. ANOVA ―― まず1回で「どこかに差があるか」を判定

分散分析（ANOVA）は、3群以上を**一度の検定**で「**どこかに差があるか**」だけを判定する。検定を繰り返さないので偽陽性が増えない。

3学部のテスト点で実行（`scipy.stats.f_oneway`）。

In [ ]:
経済 = df[df["学部"]=="経済学部"]["テスト点"]
文   = df[df["学部"]=="文学部"]["テスト点"]
社福 = df[df["学部"]=="社会福祉学部"]["テスト点"]
for name, g in [("経済",経済),("文",文),("社福",社福)]:
    print(f"{name}学部: 平均 {g.mean():.1f}点 (n={len(g)})")

F, p = stats.f_oneway(経済, 文, 社福)
print(f"\nANOVA: F値 = {F:.2f}, p値 = {p:.3f}")
if p < 0.05:
    print("→ p<0.05。どこかの学部間に差がありそう（次に事後検定でどのペアか調べる）")
else:
    print("→ p≥0.05。見かけの平均差（58.5 vs 56.4）はあるが、ばらつきを考えると")
    print("  『学部によってテスト点に差がある』とは言えない。")

北辰大では **F=1.42, p=0.243** で、平均は少し違って見える（58.5 / 56.4 / 56.8）が、**群内のばらつきに比べれば誤差の範囲**で、差があるとは言えなかった。

### F値の気持ち

ANOVAは2種類のばらつきを比べる。

$$ F = \frac{群間のばらつき（グループの平均どうしの違い）}{群内のばらつき（同じグループ内の個人差）} $$

- F が大きい ＝ グループ間の差が、個人差に比べて大きい → 「差がある」
- F が1前後 ＝ グループの違いは個人差の範囲 → 「差があるとは言えない」（今回）

---
## 4. ANOVAが有意だったら ―― 事後検定と補正

ANOVAが教えるのは「**どこかに**差がある」まで。**どのペアか**は別途調べる（事後検定）。その際も検定を繰り返すので、**有意水準を厳しくする補正**をかける。

- **ボンフェローニ補正**：α を検定回数で割る。例：5群=10ペアなら α=0.05/10=0.005 を各検定の基準にする。これで全体の偽陽性を5%に抑える。

> ❌ よくある誤り：「ANOVAが有意 ＝ すべての群が互いに違う」。
> 正しくは「**どこかに**差がある」だけ。どのペアかは事後検定で確かめる。

---
## 今日のまとめ

| 概念 | ひとこと |
|---|---|
| 多重比較の罠 | 検定を繰り返すほど、まぐれの有意（偽陽性）が増える |
| ANOVA | 3群以上を1回で「どこかに差があるか」判定。偽陽性が増えない |
| F値 | 群間のばらつき ÷ 群内のばらつき。大きいほど差あり |
| 事後検定＋補正 | どのペアかは別途。ボンフェローニ補正で α を回数で割る |
| ❌ 誤り | 全ペアt検定でOK／ANOVA有意＝全群が違う |

> **群が3つ以上なら、t検定を繰り返さない。まずANOVAで一括判定。**
> 「有意」は『どこかに差』であって『全部違う』ではない。

**課題（Moodle）**：あるシナリオで「t検定の繰り返し」と「ANOVA」のどちらが適切か、理由とともに判断する。